# Mission Eagle-1 — Exploration du simulateur d'atterrissage

Je suis charge de developper le pilote automatique du module Eagle-1 pour AstroDynamics. Avant de construire quoi que ce soit, je dois comprendre l'environnement de simulation qu'on va utiliser : **LunarLander-v3** de gymnasium.

L'idee : si notre agent apprend a atterrir dans ce simulateur, on pourra ensuite transposer la logique au vrai module.

**Objectif de ce notebook :**
1. Explorer l'environnement (qu'est-ce que l'agent voit ? qu'est-ce qu'il peut faire ?)
2. Tester un agent aleatoire (pour voir a quel point c'est dur)
3. Entrainer un premier modele PPO avec les parametres par defaut
4. Mesurer une performance de reference (baseline)

## Setup

On commence par installer les dependances. On utilise stable-baselines3 pour les algorithmes de RL et gymnasium pour l'environnement.

In [ ]:
# Sur Colab, decommenter ces lignes :
# !pip install stable-baselines3[extra] gymnasium[box2d]

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

## 1. Decouverte de l'environnement LunarLander-v3

Commencons par creer l'environnement et regarder ce qu'il contient.

> **Rappel gymnasium** : `gym.make()` cree une instance de l'environnement. Le `render_mode` se passe ici (pas a `.render()`). Pour l'instant on ne rend rien visuellement, on explore juste les proprietes.

In [ ]:
env = gym.make("LunarLander-v3")

print("Environnement :", env.spec.id)
print()
print("=== Espace d'OBSERVATION (ce que l'agent voit) ===")
print(f"Type  : {env.observation_space}")
print(f"Shape : {env.observation_space.shape}")
print(f"Min   : {env.observation_space.low}")
print(f"Max   : {env.observation_space.high}")
print()
print("=== Espace d'ACTION (ce que l'agent peut faire) ===")
print(f"Type             : {env.action_space}")
print(f"Nombre d'actions : {env.action_space.n}")

### Ce que l'agent percoit

L'agent recoit un vecteur de **8 valeurs** a chaque instant. En termes de mission Eagle-1 :

| Index | Variable | Analogie mission | Plage |
|-------|----------|-----------------|-------|
| 0 | Position X | Ecart horizontal par rapport a la zone d'atterrissage | [-1.5, 1.5] |
| 1 | Position Y | Altitude du module | [-1.5, 1.5] |
| 2 | Vitesse X | Derive laterale | [-5, 5] |
| 3 | Vitesse Y | Vitesse de descente | [-5, 5] |
| 4 | Angle | Inclinaison du module | [-3.14, 3.14] |
| 5 | Vitesse angulaire | Rotation en cours | [-5, 5] |
| 6 | Contact jambe gauche | Jambe gauche au sol ? | {0, 1} |
| 7 | Contact jambe droite | Jambe droite au sol ? | {0, 1} |

> **Note methodo** : L'espace d'observation est de type `Box(8,)` — c'est un espace **continu**. Chaque valeur est un nombre reel. C'est important pour le choix de l'algorithme plus tard.

### Ce que l'agent peut faire

| Action | Commande | Analogie mission |
|--------|----------|------------------|
| 0 | Ne rien faire | Chute libre |
| 1 | Moteur gauche | Correction laterale droite |
| 2 | Moteur principal | Freinage de descente |
| 3 | Moteur droit | Correction laterale gauche |

L'espace d'action est `Discrete(4)` — c'est un espace **discret** (4 choix possibles, pas de valeur intermediaire).

> **Note methodo** : La nature de l'espace d'action determine quels algorithmes on peut utiliser :
> - **Discret** (comme ici) : PPO, A2C, DQN
> - **Continu** (ex: force du moteur entre 0 et 1) : PPO, A2C, SAC, DDPG
>
> PPO fonctionne dans les deux cas, c'est pour ca qu'il est si populaire.

### Le systeme de recompenses

Comment l'environnement evalue notre pilote ? C'est crucial a comprendre :

| Evenement | Reward |
|-----------|--------|
| Atterrissage reussi sur la zone | +100 a +140 |
| Crash | -100 |
| Chaque jambe au contact du sol | +10 |
| Moteur principal allume | -0.3 par frame |
| Moteur lateral allume | -0.03 par frame |
| Chaque frame en l'air | Petit bonus/malus selon la trajectoire |

Le score total d'un episode est la somme de toutes ces recompenses. Un score de **200+** est considere comme "resolu" — l'agent sait atterrir.

> **Note methodo** : Comprendre le systeme de reward est fondamental en RL. C'est ce qui guide tout l'apprentissage. Si le reward est mal concu, l'agent apprend des comportements absurdes.

## 2. Agent aleatoire — A quel point c'est dur ?

Avant d'entrainer quoi que ce soit, voyons ce que donne un agent qui choisit ses actions au hasard. Ca nous donnera un point de reference : si notre modele fait moins bien qu'un agent aleatoire, c'est qu'on a un probleme.

> **Rappel API gymnasium** :
> - `env.reset()` retourne `(observation, info)` — pas juste `obs` comme l'ancienne API gym
> - `env.step(action)` retourne `(obs, reward, terminated, truncated, info)` — 5 valeurs, pas 4
> - `terminated` = fin logique (crash, atterrissage) / `truncated` = timeout (1000 steps)
> - `done = terminated or truncated`

In [ ]:
env = gym.make("LunarLander-v3")

N_EPISODES = 10
print(f"Test d'un agent ALEATOIRE sur {N_EPISODES} episodes")
print("=" * 55)

for ep in range(1, N_EPISODES + 1):
    obs, info = env.reset()
    total_reward = 0
    done = False
    n_steps = 0

    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated
        n_steps += 1

    statut = "Atterri" if (terminated and total_reward > 0) else "Crash"
    print(f"Episode {ep:2d} | Steps: {n_steps:4d} | Reward: {total_reward:7.1f} | {statut}")

env.close()

Bon... c'est pas brillant. L'agent aleatoire crash quasiment a chaque fois. Faisons un test plus serieux sur 50 episodes pour avoir des stats fiables.

In [ ]:
env = gym.make("LunarLander-v3")

rewards = []
for _ in range(50):
    obs, info = env.reset()
    total = 0
    done = False

    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total += reward
        done = terminated or truncated

    rewards.append(total)

env.close()

print(f"Agent aleatoire sur 50 episodes :")
print(f"  Reward moyen  : {np.mean(rewards):.1f}")
print(f"  Ecart-type    : {np.std(rewards):.1f}")
print(f"  Min           : {np.min(rewards):.1f}")
print(f"  Max           : {np.max(rewards):.1f}")
print(f"  (rappel : objectif = 200+)")

plt.figure(figsize=(10, 3))
plt.bar(range(len(rewards)), rewards, color="indianred", alpha=0.7)
plt.axhline(np.mean(rewards), color="black", linestyle="--", label=f"Moyenne = {np.mean(rewards):.1f}")
plt.axhline(200, color="green", linestyle="--", label="Objectif = 200")
plt.xlabel("Episode")
plt.ylabel("Reward total")
plt.title("Agent aleatoire sur LunarLander-v3")
plt.legend()
plt.tight_layout()
plt.show()

Comme prevu, l'agent aleatoire obtient un score moyen autour de **-150**. C'est tres loin de l'objectif de 200. On a notre baseline : tout ce qui fait mieux que -150 est deja un progres, mais on vise 200+.

## 3. Premier modele PPO — Parametres par defaut

On va entrainer un premier modele avec **PPO** (Proximal Policy Optimization) en utilisant les parametres par defaut de stable-baselines3.

### Pourquoi PPO ?
- C'est l'algorithme le plus polyvalent : il fonctionne sur des espaces discrets ET continus
- Il est stable et converge bien (moins de surprises que DQN)
- C'est le choix par defaut recommande par stable-baselines3

### Rappel : comment fonctionne PPO (en bref)

| Concept | Explication |
|---------|-------------|
| **Policy** | Le "cerveau" de l'agent — un reseau de neurones qui prend une observation et sort une action |
| **On-policy** | PPO apprend uniquement a partir de l'experience qu'il vient de collecter (pas de replay buffer) |
| **Clipping** | PPO limite les mises a jour trop brutales de la policy — ca le rend stable |
| **MlpPolicy** | Policy basee sur un perceptron multicouche (reseau fully-connected). Adapte aux observations vectorielles |

On garde 100 000 timesteps pour commencer — c'est rapide et ca suffit pour voir si ca apprend.

In [ ]:
env = gym.make("LunarLander-v3")

model = PPO("MlpPolicy", env, verbose=1)

print("Entrainement PPO — 100k timesteps (parametres par defaut)")
print("=" * 55)
model.learn(total_timesteps=100_000)

env.close()
print("\nEntrainement termine !")

### Evaluation du modele baseline

On utilise `evaluate_policy` de stable-baselines3. C'est plus propre que de faire la boucle a la main : ca gere automatiquement les resets et calcule la moyenne + ecart-type.

> **Note methodo** : On evalue sur **50 episodes** minimum pour avoir une mesure fiable. Un seul episode ne veut rien dire en RL — il y a beaucoup de variance.

In [ ]:
eval_env = gym.make("LunarLander-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=50)

print(f"PPO Baseline (100k steps, parametres par defaut)")
print(f"  Reward moyen  : {mean_reward:.1f}")
print(f"  Ecart-type    : {std_reward:.1f}")
print(f"  Objectif      : 200+")
print(f"  Verdict       : {'Objectif atteint !' if mean_reward > 200 else 'Pas encore... il faudra optimiser.'}")

eval_env.close()

In [ ]:
model.save("models/ppo_baseline")
print("Modele sauvegarde dans models/ppo_baseline.zip")

## Bilan

| Metrique | Agent aleatoire | PPO baseline |
|----------|-----------------|--------------|
| Reward moyen | ~-150 | A remplir apres execution |
| Ecart-type | ~80 | A remplir apres execution |
| Objectif 200+ | Non | A verifier |

On a maintenant notre point de depart. Le PPO par defaut fait deja beaucoup mieux que l'agent aleatoire, mais il n'atteint probablement pas encore 200 de maniere constante.

**Prochaine etape** : optimiser les hyperparametres de PPO pour depasser le seuil de 200 -> notebook `02_optimisation_ppo.ipynb`